In [4]:
# The corrected code for your final Colab cell
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import sys
import os
from google.colab import userdata

# --- 1. [FIX] Initialize token to None ---
# This prevents the NameError if the try block fails
NGROK_TOKEN = None

# --- 2. LOAD ALL SECRETS & SET ENVIRONMENT VARIABLES ---
try:
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    os.environ['VT_API_KEY'] = userdata.get('VT_API_KEY')
    NGROK_TOKEN = userdata.get('NGROK_TOKEN') # Load NGROK token

    if not NGROK_TOKEN:
        print("[!] ERROR: NGROK_TOKEN not found in Colab Secrets. Please add it.")

    print("[*] Successfully loaded API keys from Colab Secrets.")
except Exception as e:
    print(f"[!] Could not load Colab secrets: {e}")

# --- 3. FIX PYTHON'S IMPORT PATH ---
# Add your project's root directory to Python's "address book"
project_path = "/content/drive/MyDrive/Colab Notebooks/project folder 2/phishing_detection_api"
if project_path not in sys.path:
    sys.path.insert(0, project_path)

# --- 4. AUTHENTICATE NGROK ---
# This check is now safe, even if the 'try' block failed
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
else:
    print("[!] Cannot authenticate ngrok. Token not found.")

# --- 5. APPLY NEST_ASYNCIO ---
# This patches the event loop to allow Uvicorn to run
nest_asyncio.apply()

# --- 6. START NGROK TUNNEL ---
try:
    public_url = ngrok.connect(8000)
    print("Your public URL is:", public_url)
except Exception as e:
    print(f"Error starting ngrok: {e}. Check your NGROK_TOKEN.")

# --- 7. START THE SERVER (THE CORRECT WAY) ---
# We must run uvicorn.Server.serve() as an awaitable
# instead of the blocking uvicorn.run() to avoid the loop conflict.

# Create the configuration for the server
config = uvicorn.Config("phishing_detector.main:app", host="0.0.0.0", port=8000)

# Create the server instance
server = uvicorn.Server(config=config)

# Run the server using await server.serve()
# This will hook into Colab's existing, patched event loop
await server.serve()

[*] Successfully loaded API keys from Colab Secrets.
Your public URL is: NgrokTunnel: "https://kate-subsistent-distractively.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [1012]
INFO:     Waiting for application startup.


[*] Attempting to load all Layer 4 models into memory...


Device set to use cpu
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


    -> XGBoost model expects 87 features.
✅ All models loaded successfully.

--- AI Layer 4 Debug Info ---
XGBoost Phishing Probability: 0.33
BERT Phishing Probability:    1.00
CNN-LSTM Phishing Probability:0.99
Meta-Learner Final Probability: 0.39
INFO:     2401:4900:a1c7:c1b2:3475:7fd8:d4a3:d75e:0 - "POST /analyze HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1012]
